<a href="https://colab.research.google.com/github/PedroCorreia3880/PedroCorreia3880.github.io/blob/main/IAC_PL04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IAC · PL04 · Explicar e desconfiar

**Inteligência Artificial em Cibersegurança** · CTeSP Cibersegurança · ISLA Gaia
T04 · 23 de Outubro de 2026 · Mário Amorim

---

Hoje o detector olha para endereços (URLs) em vez de mensagens. Vais treinar uma regressão logística
e uma árvore de decisão, ler o que aprenderam, explicar uma decisão concreta e, no fim, pôr os dois
à prova com endereços que usas todos os dias.

**Antes de começar:** Ficheiro → Guardar uma cópia no Drive.

**Os endereços de phishing estão desarmados** (`hxxp` e `[.]`), para não serem clicáveis.
O caderno arma-os só por dentro, para calcular as características. Nunca copies um destes endereços para o navegador.

## Parte 0 · Os dados

In [ ]:
import os, re
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split

if not os.path.exists('urls-desarmado.csv'):
    from google.colab import files
    files.upload()                      # escolhe o ficheiro urls-desarmado.csv do Moodle

d = pd.read_csv('urls-desarmado.csv')
d['url_real'] = d.url.str.replace('hxxp', 'http', regex=False).str.replace('[.]', '.', regex=False)
d['y'] = (d.classe == 'phishing').astype(int)
print(len(d), 'endereços ·', d.y.sum(), 'de phishing')
d[['classe', 'url']].sample(6, random_state=1)

In [ ]:
# a mesma divisão de sempre: 70% treino, 30% teste, estratificada, semente 42
treino, teste = train_test_split(d, test_size=0.3, random_state=42, stratify=d.y)

# do treino separa-se ainda uma parte para VALIDAÇÃO: é com ela que se escolhem o limiar e a profundidade
sub, valid = train_test_split(treino, test_size=0.25, random_state=0, stratify=treino.y)
print('treino:', len(treino), '(sub-treino', len(sub), '+ validação', len(valid), ') · teste:', len(teste))

from sklearn.metrics import confusion_matrix, f1_score
def mede(nome, y_real, y_prev):
    vn, fp, fn, vp = confusion_matrix(y_real, y_prev).ravel()
    return {'modelo': nome, 'FP (legítimos bloqueados)': fp, 'FN (phishing que passou)': fn,
            'F1': round(f1_score(y_real, y_prev), 3)}
resultados = []

## Parte A · Características (15 min)

### A1 · As nove da literatura

Cada endereço passa a ser uma linha de nove números. Lê a função: é curta de propósito.

In [ ]:
ISCO = ['login', 'signin', 'verify', 'secure', 'account', 'update', 'confirm', 'bank', 'wallet', 'paypal']

def caracteristicas(u):
    p = urlparse(u)
    h = p.hostname or ''
    return {
        'comprimento':    len(u),
        'pontos_dominio': h.count('.'),
        'hifens':         u.count('-'),
        'digitos':        sum(c.isdigit() for c in u),
        'tem_ip':         int(bool(re.fullmatch(r'[\d.]+', h))),
        'https':          int(p.scheme == 'https'),
        'arroba':         int('@' in u),
        'caminho':        len(p.path.strip('/')) + len(p.query),
        'palavras_isco':  sum(w in u.lower() for w in ISCO),
    }

def tabela(urls, funcao=caracteristicas):
    return pd.DataFrame([funcao(u) for u in urls])

tabela(d.url_real).groupby(d.classe.values).mean().T.round(2)

> **Regista:** as três características em que as médias das duas classes são mais diferentes.
>
> **Pensa:** há alguma coluna em que uma das classes tenha sempre o mesmo valor?

### A2 · Uma característica tua

Na PL01, Parte D2, propuseste cinco características. Escolhe **uma que não esteja na lista acima** e
programa-a aqui. A função recebe o endereço `u` (texto) e devolve um número.

Exemplos de coisas que se conseguem calcular só com o texto: quantas barras `/` tem, se tem `%`,
se o domínio tem mais de três partes, quantas vogais tem o domínio, se aparece o nome de uma marca...

In [ ]:
def minha(u):
    # escreve aqui a tua característica; tem de devolver um número
    return 0

NOME_MINHA = 'a_minha'     # dá-lhe um nome curto
pd.DataFrame({NOME_MINHA: d.url_real.map(minha), 'classe': d.classe}).groupby('classe').mean()

> **Regista:** a tua característica, em português, e a média em cada classe. Achas que vai ajudar?

In [ ]:
def com_a_minha(u):
    f = caracteristicas(u)
    f[NOME_MINHA] = minha(u)
    return f

Xsub, Xval, Xtr, Xte = (tabela(x.url_real, com_a_minha) for x in (sub, valid, treino, teste))
print('Colunas que o modelo vai ver:', list(Xtr.columns))

## Parte B · Regressão logística (20 min)

### B1 · Treinar e medir

As características são postas na mesma escala antes do treino (`StandardScaler`), para que os pesos se possam comparar entre si.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, treino.y)
resultados.append(mede('Regressão logística (limiar 0,5)', teste.y, rl.predict(Xte)))
pd.DataFrame(resultados)

### B2 · Os pesos

In [ ]:
pesos = pd.Series(rl[-1].coef_[0], index=Xtr.columns).sort_values()
pesos.plot.barh(color=['#BB9184' if v < 0 else '#762023' for v in pesos], figsize=(8, 4))
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('peso: positivo empurra para phishing, negativo para legítimo')
plt.show()
pesos.round(2)

> **Responde:**
> 1. Qual a característica que mais empurra para phishing? E para legítimo?
> 2. Que peso teve a tua característica? Ajudou?
> 3. O `https` empurra para legítimo. Em 2026, quase todo o phishing usa https. Porque é que o modelo aprendeu isto? (Volta à A1.)

### B3 · O histograma das probabilidades

Lê o gráfico eixo a eixo antes de responder:
- **eixo horizontal:** a probabilidade de phishing que o modelo deu a cada endereço do teste, de 0 a 1;
- **eixo vertical:** quantos endereços caíram em cada intervalo;
- **cores:** a classe verdadeira. A linha tracejada é o limiar.

In [ ]:
prob_te = rl.predict_proba(Xte)[:, 1]
bins = np.linspace(0, 1, 21)
plt.figure(figsize=(9, 4))
plt.hist(prob_te[teste.y.values == 0], bins=bins, color='#BB9184', label='legítimos')
plt.hist(prob_te[teste.y.values == 1], bins=bins, color='#762023', alpha=0.75, label='phishing')
plt.axvline(0.5, color='black', ls='--'); plt.text(0.51, 100, 'limiar 0,5')
plt.xlabel('probabilidade de phishing'); plt.ylabel('número de endereços'); plt.legend(); plt.show()

> **Responde:** onde estão, no gráfico, os falsos negativos? E os falsos positivos? Usa as palavras «cor» e «lado da linha».

### B4 · Escolher o limiar, sem tocar no teste

Cenário: o filtro de email da escola. Deixar passar um phishing custa **5** (uma conta comprometida).
Bloquear um email legítimo custa **1** (alguém reclama e é libertado). O limiar escolhe-se na **validação**.

In [ ]:
rl_v = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xsub, sub.y)
prob_v = rl_v.predict_proba(Xval)[:, 1]
linhas = []
for lim in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    prev = (prob_v >= lim).astype(int)
    vn, fp, fn, vp = confusion_matrix(valid.y, prev).ravel()
    linhas.append({'limiar': lim, 'FP': fp, 'FN': fn, 'custo': fp * 1 + fn * 5})
pd.DataFrame(linhas)

In [ ]:
LIMIAR = 0.5     # muda para o limiar com menor custo na validação

prev_te = (rl.predict_proba(Xte)[:, 1] >= LIMIAR).astype(int)
resultados.append(mede(f'Regressão logística (limiar {LIMIAR})', teste.y, prev_te))
pd.DataFrame(resultados)

> **Regista** o limiar escolhido, o custo na validação e o FP, FN e F1 no teste.
> **Responde** numa linha: porque é que o limiar não se escolhe com o teste?

## Parte C · A árvore de decisão (20 min)

### C1 · Quantas perguntas? Escolher com a validação

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

linhas = []
for prof in [1, 2, 3, 4, 5, 8, None]:
    a = DecisionTreeClassifier(max_depth=prof, random_state=0).fit(Xsub, sub.y)
    linhas.append({'profundidade': prof if prof else 'sem limite', 'folhas': a.get_n_leaves(),
                   'exactidão sub-treino': round(a.score(Xsub, sub.y), 3),
                   'exactidão validação': round(a.score(Xval, valid.y), 3)})
pd.DataFrame(linhas)

> **Escolhe** a profundidade. Critério: a mais pequena cuja validação já não melhora de forma clara.
> Uma árvore que se lê vale mais do que uma décima a mais.

In [ ]:
PROF = 3     # muda para a profundidade escolhida

arvore = DecisionTreeClassifier(max_depth=PROF, random_state=0).fit(Xtr, treino.y)
resultados.append(mede(f'Árvore, profundidade {PROF}', teste.y, arvore.predict(Xte)))
pd.DataFrame(resultados)

### C2 · Desenhar a árvore

In [ ]:
plt.figure(figsize=(18, 7))
plot_tree(arvore, feature_names=list(Xtr.columns), class_names=['legítimo', 'phishing'],
          filled=True, rounded=True, impurity=False, max_depth=3, fontsize=9)
plt.show()

> **Responde:** qual é a primeira pergunta da árvore? Porque é que a árvore começou por essa e não por outra?

### C3 · Explicar uma decisão

A função abaixo mostra o caminho que um endereço percorre na árvore. É a explicação da decisão.

In [ ]:
def caminho(url_real):
    x = tabela([url_real], com_a_minha)
    no = arvore.decision_path(x).indices
    t = arvore.tree_
    for n in no[:-1]:
        col = x.columns[t.feature[n]]; val = x.iloc[0][col]
        lado = '<=' if val <= t.threshold[n] else '>'
        print(f'  {col} = {val}   ({lado} {t.threshold[n]:.1f})')
    print('  decisão:', ['legítimo', 'phishing'][arvore.predict(x)[0]])

caminho('https://github.com/scikit-learn/scikit-learn/issues')

> **Escreve** a explicação desta decisão numa frase que um utilizador percebesse. A explicação faz sentido?

## Parte D · O teste da realidade (20 min)

### D1 · Os teus endereços

Escreve **dez endereços legítimos** que usas de verdade: o Moodle, o teu email, um vídeo, um artigo, o Colab.
**Só legítimos.** Copia-os da barra do navegador, completos.

In [ ]:
meus = [
    'https://pt.wikipedia.org/wiki/Phishing',
    'https://colab.research.google.com/',
    # acrescenta aqui oito endereços teus, entre plicas e separados por vírgulas
]

In [ ]:
Xm = tabela(meus, com_a_minha)
r = pd.DataFrame({'endereço': meus,
                  'prob. regressão': rl.predict_proba(Xm)[:, 1].round(2),
                  'árvore': np.where(arvore.predict(Xm) == 1, 'bloqueia', 'passa')})
print('Regressão logística (limiar', LIMIAR, ') bloqueia', int((r['prob. regressão'] >= LIMIAR).sum()), 'de', len(meus))
print('Árvore bloqueia', int((r['árvore'] == 'bloqueia').sum()), 'de', len(meus))
r

> **Regista** quantos dos teus endereços cada modelo bloqueia. Compara com o F1 que mediste no teste.

### D2 · Ir aos dados

In [ ]:
leg = d[d.y == 0]
print('Legítimos que começam por https://www. :', round(leg.url_real.str.startswith('https://www.').mean() * 100, 1), '%')
print('Legítimos com caminho                  :', round((tabela(leg.url_real).caminho > 0).mean() * 100, 1), '%')
phi = d[d.y == 1]
print('Phishing que começa por https://www.   :', round(phi.url_real.str.startswith('https://www.').mean() * 100, 1), '%')
print('Phishing com caminho                   :', round((tabela(phi.url_real).caminho > 0).mean() * 100, 1), '%')

> **Responde:** o que é que o modelo aprendeu de facto? Porque é que os teus endereços são bloqueados?

### D3 · A versão honesta: só o domínio

Com estes dados, a única comparação justa é a que olha só para o nome do domínio, sem `https`, sem `www` e sem caminho.

In [ ]:
TLD_RAROS = {'xyz', 'top', 'info', 'tk', 'ml', 'ga', 'cf', 'gq', 'buzz', 'online', 'site', 'biz', 'club', 'click', 'link', 'live', 'shop'}
ALOJAMENTO = ['firebaseapp', 'web.app', 'weebly', 'wixsite', '000webhost', 'github.io', 'blogspot', 'glitch.me',
              'vercel.app', 'netlify.app', 'r2.dev', 'ipfs', 'fleek', 'pages.dev', 'godaddysites', 'workers.dev']

def dominio(u):
    h = (urlparse(u).hostname or '').removeprefix('www.')
    tld = h.rsplit('.', 1)[-1]
    return {'comp_dominio': len(h), 'pontos': h.count('.'), 'hifens': h.count('-'),
            'digitos': sum(c.isdigit() for c in h), 'tld_raro': int(tld in TLD_RAROS),
            'alojamento': int(any(s in h for s in ALOJAMENTO))}

Dtr, Dte, Dm = tabela(treino.url_real, dominio), tabela(teste.url_real, dominio), tabela(meus, dominio)
rl_dom = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Dtr, treino.y)
resultados.append(mede('Regressão logística, só domínio (0,5)', teste.y, rl_dom.predict(Dte)))
print('Dos teus endereços, bloqueia', int(rl_dom.predict(Dm).sum()), 'de', len(meus))
pd.DataFrame(resultados)

> **Responde:**
> 1. O F1 no teste subiu ou desceu em relação à B1? E quantos dos teus endereços bloqueia agora?
> 2. Qual dos dois detectores punhas em produção, e qual o número que reportavas num relatório? Justifica em três linhas.

## Parte E · Opcional: o sinal de alarme

Acrescenta uma característica sem significado nenhum: «o endereço começa por www?». Mede outra vez.

In [ ]:
def com_www(u):
    f = caracteristicas(u); f['www'] = int((urlparse(u).hostname or '').startswith('www.')); return f

Wtr, Wte = tabela(treino.url_real, com_www), tabela(teste.url_real, com_www)
rl_w = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Wtr, treino.y)
a1 = DecisionTreeClassifier(max_depth=1, random_state=0).fit(Wtr, treino.y)
print(mede('Regressão logística + www', teste.y, rl_w.predict(Wte)))
print('A árvore com uma só pergunta escolhe:', Wtr.columns[a1.tree_.feature[0]])

> **Responde:** o que aconteceu ao F1? O que é que isso te diz sobre o conjunto de dados, e o que farias se te acontecesse no Trabalho 1?

---

### O que entregar no Moodle, até quinta-feira, 29 de Outubro, às 23h59

1. Este caderno, com as células corridas e os valores registados nos sítios pedidos (PDF ou ligação partilhada com permissão de leitura).
2. Meia página com as respostas da A2, B2, B3, B4, C1, C2, C3, D1, D2 e D3. Se fizeste a Parte E, a resposta.

**Lembra-te:** os endereços de phishing não se copiam para o navegador.